<a href="https://colab.research.google.com/github/Saharochek98/Statistics-of-Stochastic-Processes/blob/main/03_sde_numerical_schemes_and_convergence_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import  ncx2, ecdf, norm, gamma, ncx2
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
class StochasticProcess:
    def __init__(self, params, T, Nx, Nt, init_state = None):
        self.params = params
        self.T = T
        self.Nx = Nx # Number of trajectories
        self.Nt = Nt # Number of discretization steps
        self.init_state = init_state

        #Wiener Process by default
        self.bxt = lambda x: 0
        self.sigmaxt = lambda x: 1
        self.bxt_x = lambda x: 0
        self.bxt_xx = lambda x: 0
        self.sigmaxt_x = lambda x: 0
        self.sigmaxt_xx = lambda x: 0

    def DefaultInitState(self):
        return np.zeros(self.Nx)

    def SampleFromDensity(self, x0, t, t0):
        result = np.zeros(self.Nx)
        for k in range(0, self.Nx):
            result[k] = x0[k] + np.random.normal(0, 1) * np.sqrt(t - t0)
        return result

    def PathGenerator(self):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = self.SampleFromDensity(x_data[i - 1], t_data[i], t_data[i - 1])
        return t_data, x_data

    def ExactSolution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + dwt[i - 1]
        return t_data, x_data

    def EulerSolution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + self.bxt(x_data[i - 1]) * dt + self.sigmaxt(x_data[i - 1]) * dwt[i - 1]
        return t_data, x_data

    def Milstein1Solution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + self.bxt(x_data[i - 1]) * dt + self.sigmaxt(x_data[i - 1]) * dwt[i - 1] + \
                        1/2 * self.sigmaxt(x_data[i - 1]) * self.sigmaxt_x(x_data[i - 1]) * (dwt[i - 1]**2 - dt)
        return t_data, x_data

    def Milstein2Solution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] + (self.bxt(x_data[i - 1]) - 1/2 * self.sigmaxt(x_data[i - 1]) * self.sigmaxt_x(x_data[i - 1])) * dt +\
                        self.sigmaxt(x_data[i - 1]) * dwt[i - 1] + 1/2 * self.sigmaxt(x_data[i - 1]) * self.sigmaxt_x(x_data[i - 1]) * dwt[i - 1]**2 +\
                        (1/2 * self.bxt(x_data[i - 1]) * self.sigmaxt_x(x_data[i - 1]) + 1/2 * self.bxt_x(x_data[i - 1]) * self.sigmaxt(x_data[i - 1]) +\
                        1/4 * self.sigmaxt(x_data[i - 1])**2 * self.sigmaxt_xx(x_data[i - 1])) * dt * dwt[i - 1] +\
                        (1/2 * self.bxt(x_data[i - 1]) * self.bxt_x(x_data[i - 1]) + 1/4 * self.bxt_xx(x_data[i - 1]) * self.sigmaxt(x_data[i - 1])**2) * dt**2
        return t_data, x_data

    def PredictorCorrectorSolution(self, dwt, eta = 1/2, alpha = 1/2):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            yw = x_data[i - 1] + self.bxt(x_data[i - 1]) * dt + self.sigmaxt(x_data[i - 1]) * dwt[i - 1]
            bwim1 = self.bxt(x_data[i - 1]) - eta * self.sigmaxt(x_data[i - 1]) * self.sigmaxt_x(x_data[i - 1])
            bwi = self.bxt(yw) - eta * self.sigmaxt(yw) * self.sigmaxt_x(yw)
            x_data[i] = x_data[i - 1] + alpha * bwi * dt + (1 - alpha) * bwim1 * dt +\
                        eta * self.sigmaxt(yw) * dwt[i - 1] + (1 - eta) * self.sigmaxt(x_data[i - 1]) * dwt[i - 1]
        return t_data, x_data

    def KPSSolution(self, dwt):
       dt = self.T / self.Nt
       t_data = np.linspace(0, self.T, self.Nt + 1)
       x_data = np.zeros((self.Nt + 1, self.Nx))

       if self.init_state is None:
           x_data[0] = self.DefaultInitState()
       else:
           x_data[0] = self.init_state

       for i in range(1, self.Nt + 1):
           x_pred = x_data[i-1] + self.bxt(x_data[i-1]) * dt + self.sigmaxt(x_data[i-1]) * dwt[i-1]
           x_data[i] = x_data[i-1] + 0.5 * (self.bxt(x_pred) + self.bxt(x_data[i-1])) * dt + \
                       self.sigmaxt(x_data[i-1]) * dwt[i-1]

       return t_data, x_data


class OUprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

        # params = [theta, mu, sigma]
        self.bxt = lambda x: self.params[0] * (self.params[1] - x)
        self.sigmaxt = lambda x: self.params[2]
        self.bxt_x = lambda x: -self.params[0]

    def DefaultInitState(self):
        return self.params[1] * np.ones(self.Nx)

    def StationaryState(self):
        theta = self.params[0]
        mu = self.params[1]
        sigma = self.params[2]

        result = np.zeros(self.Nx)

        for j in range(0, self.Nx):
            m = mu
            v = sigma**2 / (2 * theta)
            result[j] = np.random.normal(m, np.sqrt(v))
        return result

    def SampleFromDensity(self, x0, t, t0):
        theta = self.params[0]
        mu = self.params[1]
        sigma = self.params[2]

        result = np.zeros(self.Nx)

        for j in range(0, self.Nx):
            m = mu + (x0[j] - mu) * np.exp(-theta * (t - t0))
            v = sigma**2 / (2 * theta) * (1 - np.exp(-2 * theta * (t - t0)))
            result[j] = np.random.normal(m, np.sqrt(v))
        return result

    def ExactSolution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        theta = self.params[0]
        mu = self.params[1]
        sigma = self.params[2]

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        Ito_integral_sum = np.zeros(self.Nx)
        Determinated_part = mu + np.exp(-theta * t_data)[:, None] * (x_data[0] - mu)[None, :]
        for i in range(1, self.Nt + 1):
            Ito_integral_sum += sigma * np.exp(theta * t_data[i - 1]) * dwt[i - 1]
            x_data[i] = Determinated_part[i] + Ito_integral_sum * np.exp(-theta * t_data[i])
        return t_data, x_data


class CIRprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

        # params = [theta1, theta2, theta3]
        self.bxt = lambda x: self.params[0] - self.params[1] * x
        self.sigmaxt = lambda x: self.params[2] * np.sqrt(np.maximum(x, 0))
        self.bxt_x = lambda x: - self.params[1]
        self.sigmaxt_x = lambda x: self.params[2] / (2 * np.sqrt(np.maximum(x, 1e-10)))
        self.sigmaxt_xx = lambda x: -self.params[2] / (4 * np.maximum(x, 1e-10)**1.5)

    def DefaultInitState(self):
        return self.params[0] / self.params[1] * np.ones(self.Nx)

    def StationaryState(self):
        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        k = theta2
        theta = theta1 / theta2
        sigma = theta3

        result = np.zeros(self.Nx)

        for j in range(0, self.Nx):
            c = 2 * k / (sigma**2)
            u = 0
            q = 2 * k * theta / sigma**2 - 1
            result[j] = ncx2.rvs(2 * q + 2, 2 * u) / (2 * c)
        return result

    def SampleFromDensity(self, x0, t, t0):
        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        k = theta2
        theta = theta1 / theta2
        sigma = theta3

        result = np.zeros(self.Nx)

        for j in range(0, self.Nx):
            c = 2 * k / (sigma**2 * (1 - np.exp(-k * (t - t0))))
            u = c * x0[j] * np.exp(-k * (t - t0))
            #v = c * x
            q = 2 * k * theta / sigma**2 - 1
            result[j] = ncx2.rvs(2 * q + 2, 2 * u) / (2 * c)
        return result

    def ExactSolution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        theta1 = self.params[0]
        theta2 = self.params[1]
        theta3 = self.params[2]

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        Ito_integral_sum = np.zeros(self.Nx)
        Determinated_part = (x_data[0] - theta1 / theta2)[None, :] * np.exp(-theta2 * t_data)[:, None] + theta1/theta2
        for i in range(1, self.Nt + 1):
            Ito_integral_sum += theta3 * np.exp(theta2 * t_data[i - 1]) * np.sqrt(np.maximum(x_data[i - 1], 0)) * dwt[i - 1]
            x_data[i] = Determinated_part[i] + Ito_integral_sum * np.exp(-theta2 * t_data[i])
        return t_data, x_data


class BSMprocess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state = None):
        super().__init__(params, T, Nx, Nt, init_state)

        # params = [theta1, theta2] or [r, sigma]
        self.bxt = lambda x: self.params[0] * x
        self.sigmaxt = lambda x: self.params[1] * x
        self.bxt_x = lambda x: self.params[0]
        self.sigmaxt_x = lambda x: self.params[1]

    def DefaultInitState(self):
        return np.ones(self.Nx)

    def SampleFromDensity(self, x0, t, t0):
        r = self.params[0]
        sigma = self.params[1]

        result = np.zeros(self.Nx)

        m = (r - 1/2 * sigma**2) * (t - t0)
        v = sigma**2 * (t - t0)

        for j in range(0, self.Nx):
            result[j] = x0[j] * np.random.lognormal(m, np.sqrt(v))
        return result

    def ExactSolution(self, dwt):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        r = self.params[0]
        sigma = self.params[1]

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            x_data[i] = x_data[i - 1] * np.exp((r - sigma**2 / 2) * dt + sigma * dwt[i - 1])
        return t_data, x_data

Exercise  1. Suppose we have a   Brownian motion  defined by
 $$
 dX_t = \theta_1 X_tdt + \theta_2X_tdW_t, \quad X_0 = 1, \text{ with } (\theta_1, \theta_2) = (2, 0.5).
 $$
Compare approximations
*   the Euler approximation algorithm;
*   the 1st Milstein scheme;
*   the 2nd Milstein scheme;
*   the predictor-corrector method  (set default $\alpha = \eta = 1/2$);
*   KPS method

with the exact solution on $[0,1]$, levels of discretization $N=1024$.


In [ ]:
T = 1
Nx = 1
Nt = 1024

x0 = 1
init_state = x0 * np.ones(Nx)

In [ ]:
theta1 = 1
theta2 = 1/2
params = [theta1, theta2]

dwt = np.random.normal(0, 1, size = (Nt, Nx)) * np.sqrt(T/Nt)


BSMp = BSMprocess(params, T, Nx, Nt, init_state)
t_data, x_data = BSMp.EulerSolution(dwt)
plt.plot(t_data, x_data, label = 'Euler')

t_data, x_data = BSMp.ExactSolution(dwt)
plt.plot(t_data, x_data, label = 'Exact Solution')

t_data, x_data = BSMp.Milstein1Solution(dwt)
plt.plot(t_data, x_data, label = 'Milstein1')

t_data, x_data = BSMp.Milstein2Solution(dwt)
plt.plot(t_data, x_data, label = 'Milstein2')

t_data, x_data = BSMp.PredictorCorrectorSolution(dwt, 1/2, 1/2)
plt.plot(t_data, x_data, label = 'Predictor-corrector')

plt.title("GBM process")
plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

Exercise 2. Repeat the Exercise 1 for the Cox-Ingersoll-Ross process
$$
dX_t = (\theta_1 - \theta_2X_t)dt + \theta_3 \sqrt{ X_t}dW_t, X_0 = 10, (\theta_1, \theta_2, \theta_3) = (6, 3, 2).
$$

In [ ]:
theta1 = 6
theta2 = 3
theta3 = 2
params = [theta1, theta2, theta3]

dwt = np.random.normal(0, 1, size = (Nt, Nx)) * np.sqrt(T/Nt)

CIRp = CIRprocess(params, T, Nx, Nt, init_state)
t_data, x_data = CIRp.EulerSolution(dwt)
plt.plot(t_data, x_data, label = 'Euler')

t_data, x_data = CIRp.ExactSolution(dwt)
plt.plot(t_data, x_data, label = 'Exact Solution')

t_data, x_data = CIRp.Milstein1Solution(dwt)
plt.plot(t_data, x_data, label = 'Milstein1')

t_data, x_data = CIRp.Milstein2Solution(dwt)
plt.plot(t_data, x_data, label = 'Milstein2')

t_data, x_data = CIRp.PredictorCorrectorSolution(dwt, 0.4, 0.6)
plt.plot(t_data, x_data, label = 'Predictor-corrector')

plt.title("CIR process")
plt.legend(loc = 'best', fancybox = True, shadow = True)
plt.xlabel("t")
plt.ylabel("x(t)")
plt.grid(True)
plt.show()

Exercise 3. Generate 100 paths using conditional distribution of
* Ornstein-Uhlenbeck process $θ = (1, 1, 0.5)$,
* Geometric Brownian motion process $θ = (0.3, 0.1)$,
* Cox-Ingersoll-Ross process $θ = (0.4, 0.3, 0.2)$

with $X_0=0.5$.
Plot the random path for each process.

In [ ]:
Nx = 1
Nt = 300

T = 1

params_BSM = [1.0, 2.0]
params_CIR = [0.5, 0.2, 1.0]
params_OU  = [1.0, 0.5, 1.5]

init_state = 0.5 * np.ones(Nx)

OUp = OUprocess(params_OU, T, Nx, Nt, init_state)
BSMp = BSMprocess(params_BSM, T, Nx, Nt, init_state)
CIRp = CIRprocess(params_CIR, T, Nx, Nt, init_state)

t_data, x_data_OU = OUp.PathGenerator()
t_data, x_data_BSM = BSMp.PathGenerator()
t_data, x_data_CIR = CIRp.PathGenerator()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

n = np.random.randint(0, Nx)

axs[0].plot(t_data, x_data_OU[:,n], color='blue')
axs[0].set_title(f'OU process, # {n} path')
axs[0].set_xlabel('t')
axs[0].set_ylabel('x(t)')

axs[1].plot(t_data, x_data_BSM[:,n], color='orange')
axs[1].set_title(f'BSM process, # {n} path')
axs[1].set_xlabel('t')
axs[1].set_ylabel('x(t)')

axs[2].plot(t_data, x_data_CIR[:,n], color='green')
axs[2].set_title(f'CIR process, # {n} path')
axs[2].set_xlabel('t')
axs[2].set_ylabel('x(t)')

plt.tight_layout()

plt.show()

In [ ]:
# save 100 paths to files

pd.DataFrame(data = x_data_OU, index = t_data).to_csv("OUprocess.csv")

pd.DataFrame(data = x_data_BSM, index = t_data).to_csv("BSMprocess.csv")

pd.DataFrame(data = x_data_CIR, index = t_data).to_csv("CIRprocess.csv")

# Assignments


## Task 1 — Comparison of Numerical Schemes

Using the `StochasticProcess` class template, implement and compare the numerical schemes described in the assignment. Evaluate their accuracy against an appropriate reference solution and summarize the results.


In [ ]:
# KPSSolution is implemented in the StochasticProcess base class

In [ ]:
T = 1.0
N = 1024
Nx = 10000
np.random.seed(42)

params = []
process = StochasticProcess(params, T, Nx, N)
dt = T / N
dwt = np.random.normal(0, np.sqrt(dt), (N, Nx))

t_exact, x_exact = process.ExactSolution(dwt)
t_euler, x_euler = process.EulerSolution(dwt)
t_mil1, x_mil1 = process.Milstein1Solution(dwt)
t_mil2, x_mil2 = process.Milstein2Solution(dwt)
t_pc, x_pc = process.PredictorCorrectorSolution(dwt, eta=0.5, alpha=0.5)
t_kps, x_kps = process.KPSSolution(dwt)

methods = {
    'Euler–Maruyama': x_euler,
    'Milstein (first order)': x_mil1,
    'Milstein (higher-order variant)': x_mil2,
    'Predictor–corrector': x_pc,
    'KPS': x_kps
}

results = []
for name, x_approx in methods.items():
    max_error = np.max(np.abs(x_exact - x_approx))
    results.append({'Method': name, 'Maximum absolute error': max_error})

df_results = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.2e}'.format)
print(df_results)

**Interpretation:** For standard Brownian motion with constant drift zero and diffusion one, the listed schemes reduce to the same discrete Brownian increments. Their differences are at floating-point roundoff level; this experiment does **not** establish their convergence orders for general SDEs.


## Task 2 — Convergence Analysis

1. Quantify approximation errors for Euler–Maruyama, first-order Milstein, and the higher-order scheme over $N\in\{2,4,8,\ldots,32768\}$.
2. Plot error against discretization resolution and estimate the empirical log–log slopes.
3. Interpret these slopes carefully: the usual strong orders are 0.5 for Euler–Maruyama and 1.0 for Milstein under standard assumptions. The higher-order implementation does not automatically have strong order 2 for general SDEs.


In [ ]:
def task_2():
    r = 0.05
    sigma = 0.2
    params = [r, sigma]
    T = 1.0
    Nx = 100

    powers = np.arange(1, 16)
    N_values = 2 ** powers

    errors_euler = []
    errors_milstein1 = []
    errors_milstein2 = []
    dt_values = []

    print(f"{'N'}    {'dt'}   {'Err Euler'}   {'Err Mil1'}    {'Err Mil2'}")

    for Nt in N_values:
        dt = T / Nt
        dt_values.append(dt)

        process = BSMprocess(params=params, T=T, Nx=Nx, Nt=Nt)
        dwt = np.random.normal(0, np.sqrt(dt), size=(Nt, Nx))

        _, x_exact = process.ExactSolution(dwt)
        _, x_euler = process.EulerSolution(dwt)
        _, x_mil1 = process.Milstein1Solution(dwt)
        _, x_mil2 = process.Milstein2Solution(dwt)

        err_euler = np.mean(np.abs(x_exact[-1, :] - x_euler[-1, :]))
        err_mil1 = np.mean(np.abs(x_exact[-1, :] - x_mil1[-1, :]))
        err_mil2 = np.mean(np.abs(x_exact[-1, :] - x_mil2[-1, :]))

        errors_euler.append(err_euler)
        errors_milstein1.append(err_mil1)
        errors_milstein2.append(err_mil2)

        print(f"{Nt}     {dt:.6f}    {err_euler:.6e}    {err_mil1:.6e}      {err_mil2:.6e}")

    log_dt = np.log(dt_values)

    # Estimate empirical convergence orders
    dt_array = np.asarray(dt_values)

    methods = [
        ("Euler–Maruyama", errors_euler, "royalblue"),
        ("Milstein 1", errors_milstein1, "crimson"),
        ("Milstein 2", errors_milstein2, "forestgreen"),
    ]

    fig, axes = plt.subplots(
        1, 3, figsize=(15, 4.5),
        constrained_layout=True
    )

    for ax, (name, errors, color) in zip(axes, methods):
        errors = np.asarray(errors)

        # Fit: log(error) = p * log(dt) + log(C)
        slope, intercept = np.polyfit(
            np.log(dt_array),
            np.log(errors),
            1
        )

        fitted_errors = np.exp(intercept) * dt_array**slope

        ax.loglog(
            dt_array, errors,
            "o-", color=color,
            linewidth=2, markersize=5,
            label="Simulation"
        )

        ax.loglog(
            dt_array, fitted_errors,
            "--", color="black",
            alpha=0.7,
            label=f"Fitted order: {slope:.2f}"
        )

        ax.set_title(name)
        ax.set_xlabel("Time step Δt")
        ax.set_ylabel("Mean absolute error")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()

        print(f"{name}: estimated order = {slope:.3f}")

    plt.show()


    return errors_euler, errors_milstein1, errors_milstein2, N_values, dt_values

errors_euler, errors_milstein1, errors_milstein2, N_values, dt_values = task_2()

**Conclusion:**

The numerical experiments demonstrate convergence for all three stochastic numerical schemes.

The Euler–Maruyama method achieves an estimated convergence order of **0.487**, consistent with its theoretical strong convergence order of 0.5.

Milstein 1 achieves an estimated order of **0.976**, consistent with first-order strong convergence.

Milstein 2 produces the smallest absolute numerical errors across all tested time steps, with an estimated convergence order of **0.977**. Although it is more accurate in this experiment, it does not demonstrate second-order convergence.

The log-log plots show an approximately linear relationship between numerical error and time step, supporting the estimated convergence rates.

**Overall, the results demonstrate that higher numerical accuracy does not necessarily imply a higher convergence order.**

## Task 3 — Predictor–Corrector Parameter Study

The predictor–corrector method depends on averaging parameters $\alpha$ and $\eta$.

1. Examine the effect of $\alpha,\eta\in[0,1]$ on numerical accuracy.
2. Plot the error surface for the parameter grid.
3. Interpret the limitations of the selected reference and compare the parameter choices.


In [ ]:
params = [2.0, 1.0, 0.5]
T = 1.0
Nx = 500
Nt = 128

process = CIRprocess(params=params, T=T, Nx=Nx, Nt=Nt)
np.random.seed(42)
dt = T / Nt
dwt = np.random.normal(0, np.sqrt(dt), size=(Nt, Nx))

t_data = np.linspace(0, T, Nt + 1)
x_independent_reference = np.zeros((Nt + 1, Nx))
x_independent_reference[0] = process.DefaultInitState()

for i in range(1, Nt + 1):
    x_independent_reference[i] = process.SampleFromDensity(x_independent_reference[i-1], t_data[i], t_data[i-1])


alpha_values = np.linspace(0, 1, 11)
eta_values = np.linspace(0, 1, 11)
discrepancy_matrix = np.zeros((len(alpha_values), len(eta_values)))

for i, alpha in enumerate(alpha_values):
    for j, eta in enumerate(eta_values):
        _, x_pc = process.PredictorCorrectorSolution(dwt, eta=eta, alpha=alpha)

        discrepancy_matrix[i, j] = np.max(np.abs(x_independent_reference - x_pc))

A, E = np.meshgrid(alpha_values, eta_values)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(A, E, discrepancy_matrix.T, cmap='viridis', edgecolor='none', alpha=0.9)

ax.set_xlabel('alpha', fontsize=12, labelpad=10)
ax.set_ylabel('eta', fontsize=12, labelpad=10)
ax.set_zlabel('Independent-path discrepancy ', fontsize=12, labelpad=10)
ax.set_title('Predictor–corrector discrepancy across alpha and eta', fontsize=14)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

plt.tight_layout()
plt.show()

min_idx = np.unravel_index(np.argmin(discrepancy_matrix), discrepancy_matrix.shape)
opt_alpha = alpha_values[min_idx[0]]
opt_eta = eta_values[min_idx[1]]
min_error = discrepancy_matrix[min_idx]

print(f"alpha_opt = {opt_alpha}")
print(f"eta_opt   = {opt_eta}")
print(f"Minimum discrepancy = {min_error:.6f}")

**Conclusion:**

The three-dimensional surface illustrates how the discrepancy varies with the predictor–corrector parameters α and η.

The discrepancy decreases as α increases and increases as η increases. The smallest observed value occurs near α = 1 and η = 0, while the largest occurs near α = 0 and η = 1.

However, the overall variation is small, ranging approximately from 2.158 to 2.165, indicating limited sensitivity to these parameters under the selected experimental setup.

**An important limitation** is that the discrepancy is calculated using independent stochastic trajectories. Consequently, it should not be interpreted as a direct measure of strong numerical error or used to identify the most accurate parameter combination.

A more rigorous accuracy assessment would require comparing the numerical solutions against a reference solution driven by the same Brownian path.

## Task 4 — Distributional Analysis of Additional Diffusions

1. Implement the Pearson, Jacobi, radial OU, and Ahn–Gao examples.
2. Compare empirical cumulative distribution functions (ECDFs) and density histograms at $t=0.1,0.5,1.0$.
3. Mark the empirical 90th percentile on each plot.
4. Discuss model-specific state-space and tail behavior.

**Modeling caveat:** The radial OU implementation below is a reflected OU approximation, not a general radial Ornstein–Uhlenbeck diffusion. Likewise, the Pearson example uses a power-law diffusion coefficient; naming conventions vary across references.


In [ ]:
class PearsonProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        self.theta = params[0]
        self.mu = params[1]
        self.sigma = params[2]
        self.gamma = params[3]

        self.bxt = lambda x: self.theta * (self.mu - x)
        self.sigmaxt = lambda x: self.sigma * np.maximum(x, 1e-8)**self.gamma

    def DefaultInitState(self):
        return np.ones(self.Nx) * self.mu

class JacobiProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        self.theta = params[0]
        self.mu = params[1]
        self.sigma = params[2]

        self.bxt = lambda x: self.theta * (self.mu - x)
        self.sigmaxt = lambda x: self.sigma * np.sqrt(np.maximum(x * (1 - x), 0))

    def DefaultInitState(self):
        return np.ones(self.Nx) * 0.5

class RadialOUProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        self.theta = params[0]
        self.mu = params[1]
        self.sigma = params[2]

        self.bxt = lambda x: self.theta * (self.mu - x)
        self.sigmaxt = lambda x: self.sigma

    def DefaultInitState(self):
        return np.abs(np.ones(self.Nx) * self.mu)

    def PathGenerator(self):
        dt = self.T / self.Nt
        t_data = np.linspace(0, self.T, self.Nt + 1)
        x_data = np.zeros((self.Nt + 1, self.Nx))

        if self.init_state is None:
            x_data[0] = self.DefaultInitState()
        else:
            x_data[0] = self.init_state

        for i in range(1, self.Nt + 1):
            dW = np.random.normal(0, np.sqrt(dt), self.Nx)
            x_raw = x_data[i-1] + self.bxt(x_data[i-1]) * dt + self.sigmaxt(x_data[i-1]) * dW
            x_data[i] = np.abs(x_raw)
        return t_data, x_data

class AhnGaoProcess(StochasticProcess):
    def __init__(self, params, T, Nx, Nt, init_state=None):
        super().__init__(params, T, Nx, Nt, init_state)
        self.kappa = params[0]
        self.theta = params[1]
        self.sigma = params[2]

        self.bxt = lambda x: self.kappa * (self.theta - x)
        self.sigmaxt = lambda x: self.sigma * np.maximum(x, 1e-8)**1.5

    def DefaultInitState(self):
        return np.ones(self.Nx) * self.theta

def plot_process_results(process, name, times=[0.1, 0.5, 1.0]):
    Nx_plot = 5000  # Reduced from 50,000 to keep notebook memory usage manageable
    process.Nx = Nx_plot
    process.init_state = process.DefaultInitState()

    t_data, x_paths = process.PathGenerator()
    indices = [np.argmin(np.abs(t_data - t)) for t in times]

    fig, axes = plt.subplots(len(times), 2, figsize=(14, 6 * len(times)))
    fig.suptitle(f'{name} Process Analysis', fontsize=16)

    for idx, t_idx in enumerate(indices):
        t_current = times[idx]
        samples = x_paths[t_idx, :]

        q90 = np.quantile(samples, 0.9)

        ax_ecdf = axes[idx, 0]
        sorted_samples = np.sort(samples)
        cdf_values = np.arange(1, len(sorted_samples) + 1) / len(sorted_samples)
        ax_ecdf.plot(sorted_samples, cdf_values, label=f't={t_current}')
        ax_ecdf.axvline(q90, color='red', linestyle='--', label=f'90% Quantile ({q90:.3f})')
        ax_ecdf.set_title(f'ECDF at t={t_current}')
        ax_ecdf.set_xlabel('X')
        ax_ecdf.set_ylabel('Probability')
        ax_ecdf.legend()
        ax_ecdf.grid(True)

        ax_hist = axes[idx, 1]
        ax_hist.hist(samples, bins=100, density=True, alpha=0.7, color='green', label='Empirical PDF')
        ax_hist.axvline(q90, color='red', linestyle='--', label=f'90% Quantile ({q90:.3f})')
        ax_hist.set_title(f'Histogram at t={t_current}')
        ax_hist.set_xlabel('X')
        ax_hist.set_ylabel('Density')
        ax_hist.legend()
        ax_hist.grid(True)

    plt.tight_layout()
    plt.show()

T_final = 1.0
Nt_steps = 1000

p_pearson = PearsonProcess(params=[2.0, 1.0, 0.5, 0.5], T=T_final, Nx=1, Nt=Nt_steps)
plot_process_results(p_pearson, "Pearson (gamma=0.5)")
p_jacobi = JacobiProcess(params=[5.0, 0.5, 1.0], T=T_final, Nx=1, Nt=Nt_steps)
plot_process_results(p_jacobi, "Jacobi")
p_radial = RadialOUProcess(params=[2.0, 0.0, 1.0], T=T_final, Nx=1, Nt=Nt_steps)
plot_process_results(p_radial, "Radial OU")
p_ahngao = AhnGaoProcess(params=[2.0, 1.0, 0.5], T=T_final, Nx=1, Nt=Nt_steps)
plot_process_results(p_ahngao, "Ahn-Gao")

**Conclusion:**

The empirical cumulative distribution functions (ECDFs) and density histograms illustrate the evolution of the Ahn–Gao process at three time points: t = 0.1, 0.5, and 1.0.

The distributions become progressively more dispersed over time, indicating increasing uncertainty in the process values. This is reflected in the increasing 90th percentile, which rises from 1.412 at t = 0.1 to 1.911 at t = 0.5 and 2.277 at t = 1.0.

The ECDF curves become less steep as time increases, consistent with the widening distributions. Meanwhile, the centers of the distributions remain approximately stable.

**Overall**, the simulation suggests that the variability of the Ahn–Gao process increases over the observed time horizon. The results also motivate a check of the numerical scheme's ability to preserve the theoretical state-space constraints of the model.


## Task 5 — Conditional Moments and Stationary Benchmarks

1. Simulate 100 trajectories for OU with parameters $(1.0,1.0,0.5)$ and CIR with $(0.4,0.3,0.2)$, using $X_0=0.5$, $T=1$, and 300 time steps.
2. Compare terminal histograms with theoretical **stationary** densities, noting that the finite-time distributions need not be stationary.
3. Compare simulated and theoretical conditional means and variances.
4. Plot ten paths, terminal histograms, and the time evolution of the first two moments.
5. Report moment estimates at $t=0.1,0.5,1.0$ and discuss Monte Carlo uncertainty.


In [ ]:
def ou_theory(params, x0, t):
    theta, mu, sigma = params
    mean = mu + (x0 - mu) * np.exp(-theta * t)
    var = (sigma**2 / (2 * theta)) * (1 - np.exp(-2 * theta * t))
    return mean, var

def cir_theory_mean_var(params, x0, t):
    k, theta, sigma = params
    mean = theta + (x0 - theta) * np.exp(-k * t)
    var = x0 * (sigma**2 / k) * (np.exp(-k * t) - np.exp(-2 * k * t)) + \
          theta * (sigma**2 / (2 * k)) * (1 - np.exp(-k * t))**2
    return mean, var

def cir_stationary_pdf(x, params):
    k, theta, sigma = params
    shape = 2 * k * theta / (sigma**2)
    scale = sigma**2 / (2 * k)
    return gamma.pdf(x, a=shape, scale=scale)

def ou_stationary_pdf(x, params):
    theta, mu, sigma = params
    var_stat = sigma**2 / (2 * theta)
    return norm.pdf(x, loc=mu, scale=np.sqrt(var_stat))

T = 1.0
Nt = 300
Nx = 100
x0 = 0.5
dt = T / Nt
t_vec = np.linspace(0, T, Nt + 1)

params_ou = [1.0, 1.0, 0.5]
params_cir = [0.4, 0.3, 0.2]

np.random.seed(123)

dW_ou = np.random.normal(0, np.sqrt(dt), size=(Nt, Nx))
X_ou = np.zeros((Nt + 1, Nx))
X_ou[0] = x0
for i in range(Nt):
    X_ou[i+1] = X_ou[i] + params_ou[0] * (params_ou[1] - X_ou[i]) * dt + params_ou[2] * dW_ou[i]

dW_cir = np.random.normal(0, np.sqrt(dt), size=(Nt, Nx))
X_cir = np.zeros((Nt + 1, Nx))
X_cir[0] = x0
for i in range(Nt):
    drift = params_cir[0] * (params_cir[1] - X_cir[i]) * dt
    diff = params_cir[2] * np.sqrt(np.maximum(X_cir[i], 0)) * dW_cir[i]
    X_cir[i+1] = np.maximum(X_cir[i] + drift + diff, 0)

times_check = [0.1, 0.5, 1.0]

for proc_name, X, params in [("OU", X_ou, params_ou), ("CIR", X_cir, params_cir)]:
    print(f"Process: {proc_name}")
    print('Time     E_emp    E_theo    Var_emp   Var_theo')
    for t in times_check:
        idx = int(round(t / dt))
        samples = X[idx, :]

        emp_mean = np.mean(samples)
        emp_var = np.var(samples)

        if proc_name == "OU":
            theo_mean, theo_var = ou_theory(params, x0, t)
        else:
            theo_mean, theo_var = cir_theory_mean_var(params, x0, t)

        print(f"{t:.1f}      {emp_mean:.4f}      {theo_mean:.4f}      {emp_var:.4f}      {theo_var:.4f}")

def plot_panel(process_name, X, t_vec, params, theory_func, stat_pdf_func):
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), gridspec_kw={'height_ratios': [2, 2, 2]})

    ax1 = axes[0]
    indices_random = np.random.choice(Nx, 10, replace=False)
    for idx in indices_random:
        ax1.plot(t_vec, X[:, idx], alpha=0.7)
    ax1.set_title(f'{process_name} process: 10 trajectories')
    ax1.set_ylabel('X(t)')
    ax1.grid(True)

    ax2 = axes[1]
    final_values = X[-1, :]
    ax2.hist(final_values, bins=15, density=True, alpha=0.6, color='skyblue', edgecolor='black', label='Empirical histogram (t=T)')

    x_range = np.linspace(min(final_values), max(final_values), 200)
    pdf_vals = stat_pdf_func(x_range, params)
    ax2.plot(x_range, pdf_vals, 'r-', linewidth=2, label='Stationary density (long-run reference)')

    ax2.legend()
    ax2.grid(True)

    ax3 = axes[2]
    mean_theo = []
    var_theo = []
    for t in t_vec:
        m, v = theory_func(params, x0, t)
        mean_theo.append(m)
        var_theo.append(v)

    mean_emp = np.mean(X, axis=1)
    var_emp = np.var(X, axis=1)

    ax3_twin = ax3.twinx()

    line1 = ax3.plot(t_vec, mean_theo, 'b-', label='Theoretical mean')
    line2 = ax3.plot(t_vec, mean_emp, 'b--', label='Sample mean')
    ax3.set_xlabel('Time')
    ax3.set_ylabel('E', color='b')
    ax3.tick_params(axis='y', labelcolor='b')

    line3 = ax3_twin.plot(t_vec, var_theo, 'r-', label='Theoretical variance')
    line4 = ax3_twin.plot(t_vec, var_emp, 'r--', label='Sample variance')
    ax3_twin.set_ylabel('D', color='r')
    ax3_twin.tick_params(axis='y', labelcolor='r')


    lines = line1 + line2 + line3 + line4
    labels = [l.get_label() for l in lines]
    ax3.legend(lines, labels, loc='upper right')
    ax3.grid(True)

    plt.tight_layout()
    plt.show()

plot_panel("OU", X_ou, t_vec, params_ou, ou_theory, ou_stationary_pdf)
plot_panel("CIR", X_cir, t_vec, params_cir, cir_theory_mean_var, cir_stationary_pdf)

**Conclusion:**

The simulated trajectories exhibit stochastic fluctuations and a tendency to revert toward the long-term mean.

The empirical mean and variance closely follow their theoretical time-dependent counterparts, supporting the numerical implementation of the Ornstein–Uhlenbeck process.

The empirical distribution at the final simulation time differs from the stationary normal distribution because the process starts from a fixed initial condition and has not necessarily reached stationarity.

**Overall**, the results demonstrate the mean-reverting behavior of the OU process and show good agreement between the simulated and theoretical moments.